In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # put this at the very top of your script

from datasets import load_dataset, Dataset
from dataclasses import dataclass
from typing import Optional
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer
from torch.utils.data import DataLoader

from tools_llada import TopKSorter, TruthCollector, MaxCollector
from plugins_llada import \
    SaveKVPreviousPlugin_Disabled, SaveKVPreviousPlugin_Enabled,\
    CachePastKVPlugin_Disabled, CachePastKVPlugin_Enabled,\
    CacheAttnPlugin_Disabled, CacheAttnPlugin_Enabled,\
    CacheVOPlugin_Disabled, CacheVOPlugin_Enabled

from configs_llada import DiffusionConfig

from components_llada import SimpleLogitsSnapshot
from tools_llada import ConfKSorter, ConfCollectorInterface, BlockDiffusionQuotaHelper
from plugins_llada import InspectorPlugin

from dataset_llada import LIST_DATASET
from datapreprocess_llada import parse_lines_with_index, merge_subdocs, PATTEN_REG_WIKI
from dataprocess_llada import Tokenizer_wiki_simple, Collater_wiki_simple

from modeling_llada_yukai_06 import LLaDAModelLM

from tools_debug import jprint


config = DiffusionConfig(
    id_model='GSAI-ML/LLaDA-8B-Base',
    len_prompt=32,
    len_target=4*64,
    num_blocks=4,
    num_unmask_per_step=1,
    id_mask=126336,
    size_batch=1,
    device='cuda:0',
    klass_sorter=TopKSorter,
    klass_collector=TruthCollector,
    klass_save_kv_previous=SaveKVPreviousPlugin_Disabled,
    klass_cache_past_kv=CachePastKVPlugin_Enabled,
    klass_cache_attn=CacheAttnPlugin_Enabled,
    klass_cache_vo=CacheVOPlugin_Disabled
)

/home/exx/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
'''load dataset first'''
name_dataset = LIST_DATASET[1]
ds = load_dataset(*name_dataset, split='all')
docs, _ = parse_lines_with_index(PATTEN_REG_WIKI, ds['text'])
docs = docs['subdocs']

samples = []
for doc in docs:
    lines_1 = doc['texts']
    paragraph_1 = ' '.join(lines_1)
    lines_remain, titles = merge_subdocs(doc['subdocs'])
    paragraph_remain = ' '.join(lines_remain)
    prefix = paragraph_1
    target = paragraph_remain
    samples.append({'text': paragraph_1 + ' ' + paragraph_remain})
# end

ds_origin = Dataset.from_list(samples[:100])


'''initialize constant hyper-parameters'''

'''load tokenizer'''
tokenizer = AutoTokenizer.from_pretrained(
    config.id_model,
    trust_remote_code=True
)

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'
# end
assert tokenizer.pad_token_id != 126336


'''load model'''
model_kwargs = {}
model = LLaDAModelLM.from_pretrained(
    config.id_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    **model_kwargs
)

model = model.eval().to(config.device)


'''start to handle dataset based on hyper-parameter'''
len_max = config.len_prompt + config.len_target
ds = ds_origin\
    .filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)\
    .map(Tokenizer_wiki_simple(tokenizer, len_max), remove_columns=ds_origin.column_names)\
    .filter(lambda x: x["length"] >= len_max)\
    .sort("length")
# end

'''prepare dataloader'''
loader = DataLoader(
    ds,
    batch_size=config.size_batch,
    shuffle=False,                 # keep sorted order
    collate_fn=Collater_wiki_simple(len_max, config.len_prompt, config.len_target, config.id_mask),
    drop_last=False
)

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Filter: 100%|██████████| 100/100 [00:00<00:00, 22496.80 examples/s]


In [3]:
class FutureIDXSelector:
    def __init__(self, model, h=5, select_only_in_h=True):
        self.model = model
        self.select_only_in_h = select_only_in_h
        self.h = h
    # end

    def select_future_by_attn(self, attn):  # attn.shape: (1,64)
        attn_avail = (attn >0).nonzero(as_tuple=True)[1].reshape(attn.shape[0], -1)
        idx = torch.rand(attn_avail.shape, device=attn.device).argsort(dim=-1)[:, :self.h]
        return torch.gather(attn_avail, 1, idx)
    # end
# end

In [4]:
@ torch.no_grad()

def run_model_semi_cached_snapshot_query_h_attn(model, x, y, config_diffusion, *args, **kwargs):

    '''declare required variables'''
    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.klass_sorter()
    collector = config_diffusion.klass_collector()

    plugin_cache_attn = kwargs['plugin_cache_attn']
    step_refresh_remainder = kwargs['step_refresh_remainder']
    future_idx_selector = kwargs['future_idx_selector'] # budget is also here


    idx_refresh = torch.tensor([], dtype=torch.long, device=x.device)
    p_finalized = torch.zeros(x.shape, dtype=torch.float64, device=x.device)

    position_start, position_end = 0, len_prompt
    idx_denoising = torch.arange(position_start, position_end, dtype=torch.long, device=x.device)
    idx_current = torch.cat([idx_refresh, idx_denoising])
    shape_target = (x.shape[0], position_end, -1)
    logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
    snapshot = SimpleLogitsSnapshot(logits, x[:, idx_current], y[:, idx_current], id_mask)

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_block = x[:,position_start:position_end] == id_mask
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_block, size_block)

        idx_block = torch.arange(position_start, position_end, dtype=torch.long, device=x.device)
        shape_target = (x.shape[0], position_end, -1)

        for step in range(step_per_block):

            if step == 0 or step % step_refresh_remainder == 0:
                idx_denoising = idx_block

                if step == 0:
                    idx_current = torch.cat([idx_refresh, idx_denoising])   # only the first time need refresh previous
                else:
                    idx_current = idx_denoising
                # end

                logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
                logits_denoising = logits[:, -idx_denoising.shape[-1]:]

                logits_accumulated = torch.cat([snapshot.get_logits()[:, :position_start, :], logits_denoising], dim=1)
                x_accumulated = x[:, :position_end]
                y_accumulated = y[:, :position_end]
                snapshot = SimpleLogitsSnapshot(logits_accumulated, x_accumulated, y_accumulated, id_mask)
                conf_snapshot = snapshot.transform_logits(collector)
            else:
                score_attn = plugin_cache_attn.collect_attn_from_all_blocks(model)
                idx_in_attn = torch.where(idx_transform_2d.squeeze(0) == idx_current)[0]    # idx_current is now last round
                score_attn = score_attn[-1, idx_in_attn, -idx_block.shape[-1]:].squeeze(1)
                mask_mask_current_no = ~(x[:,position_start:position_end] == id_mask).view(1,-1)    # (B, K)
                score_attn.masked_fill_(mask_mask_current_no, torch.finfo(score_attn.dtype).min)

                idx_denoising = (future_idx_selector.select_future_by_attn(score_attn) + position_start).squeeze(0)
                idx_current = torch.cat([idx_refresh, idx_denoising])

                logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
                logits_transform = logits[:, -idx_denoising.shape[-1]:]

                # different here compared to step == 0
                snapshot.update_logits_(idx_denoising.unsqueeze(0), logits_transform)
                conf_snapshot = snapshot.transform_logits(collector)
                # different ends

                if future_idx_selector.select_only_in_h: #TODO: be careful of the use of scatter(shape)
                    mask_denoising_no = ~torch.isin(torch.arange(conf_snapshot.shape[-1], device=conf_snapshot.device), idx_denoising).unsqueeze(0)    # idx_denoising -> 
                    conf_snapshot.masked_fill_(mask_denoising_no, torch.finfo(conf_snapshot.dtype).min)
                # end

            # end

            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)    # truth
            num_unmask = quota_helper.get_quota(step)
            idx_transform_2d = idx_sorted_by_conf[:, :num_unmask]
            snapshot.materialize_by_idx_(idx_transform_2d, conf_snapshot)
            snapshot.update_this(1, idx_transform_2d, y=x).update_this(1, idx_transform_2d, p_finalized=p_finalized)
            idx_refresh = idx_transform_2d.squeeze(0)
        # end
    # end

    return p_finalized[:, len_prompt:]
# end

In [ ]:
import json
from tqdm import tqdm
from tools_llada import PPLCalculator, RefreshIdxHelper
from future_idx_selector import FutureIDXSelector, RandomModel, FutureIdxSelectorModelLoader

calculator_ppl = PPLCalculator()

loader_mlp = FutureIdxSelectorModelLoader(1, config.device)
## if
mlp = loader_mlp.load('mlp_attn.pt')
mlp = mlp.to(torch.bfloat16)
## else
# mlp = RandomModel()
##

future_idx_selector = FutureIDXSelector(mlp)

model\
    .fill_plugin(config.klass_cache_past_kv)\
    .fill_plugin(config.klass_save_kv_previous)\
    .fill_plugin(config.klass_cache_attn)\
    .fill_plugin(config.klass_cache_vo)

plugin_cache_past_kv = config.klass_cache_past_kv()
plugin_cache_attn = config.klass_cache_attn()

'''start the evaluation process'''
for id_batch, batch in enumerate(tqdm(loader)):
    plugin_cache_past_kv.clear(model)
    plugin_cache_attn.clear(model)

    conf = run_model_semi_cached_snapshot_query_h_attn(
        model,
        batch['ids_prompt_masked_full'].to(config.device),
        batch['ids_target_masked_full'].to(config.device),
        config,
        plugin_cache_attn=plugin_cache_attn,
        step_refresh_remainder=30,
        future_idx_selector=future_idx_selector
    )

    print(f'{calculator_ppl.cal(conf)}')
# end

  0%|          | 0/92 [00:00<?, ?it/s]

  1%|          | 1/92 [00:09<14:11,  9.36s/it]

(11.186714449774998, 0.35935515338362056)


  2%|▏         | 2/92 [00:18<13:27,  8.97s/it]

(14.22586838000857, 0.22083489652150956)


...

(14.784086234641865, 0.26007548174763284)


 99%|█████████▉| 91/92 [13:13<00:08,  8.73s/it]

(17.988396458844537, 0.24000284167336638)


100%|██████████| 92/92 [13:21<00:00,  8.72s/it]

(14.3602030224134, 0.2757115925131479)
